# Data Layer Validation

Validation of the AWS analytical data layer built from the raw Favorita datasets.

Objectives:
- validate raw and curated schemas;
- verify row counts and business grains;
- detect unintended row duplication after joins;
- validate missing-value treatments;
- confirm that curated datasets preserve the expected forecasting structure.

In [14]:
import json
import subprocess
import time

import pandas as pd

GLUE_DATABASE = "retail_demand_forecasting_raw"
CURATED_DATABASE = "retail_demand_forecasting_curated"

ATHENA_OUTPUT = (
    "s3://retail-demand-forecasting-platform-aws-zaamine/"
    "artifacts/athena-query-results/notebook/"
)


def run_athena_query(query):
    """Run an Athena query and return the result as a pandas DataFrame."""

    start_result = subprocess.run(
        [
            "aws",
            "athena",
            "start-query-execution",
            "--query-string",
            query,
            "--result-configuration",
            f"OutputLocation={ATHENA_OUTPUT}",
            "--output",
            "json",
        ],
        capture_output=True,
        text=True,
        check=True,
    )

    query_execution_id = json.loads(start_result.stdout)["QueryExecutionId"]

    while True:
        status_result = subprocess.run(
            [
                "aws",
                "athena",
                "get-query-execution",
                "--query-execution-id",
                query_execution_id,
                "--query",
                "QueryExecution.Status.State",
                "--output",
                "text",
            ],
            capture_output=True,
            text=True,
            check=True,
        )

        status = status_result.stdout.strip()

        if status in {"SUCCEEDED", "FAILED", "CANCELLED"}:
            break

        time.sleep(1)

    if status != "SUCCEEDED":
        raise RuntimeError(
            f"Athena query ended with status: {status}"
        )

    result = subprocess.run(
        [
            "aws",
            "athena",
            "get-query-results",
            "--query-execution-id",
            query_execution_id,
            "--output",
            "json",
        ],
        capture_output=True,
        text=True,
        check=True,
    )

    payload = json.loads(result.stdout)
    rows = payload["ResultSet"]["Rows"]

    columns = [
        item.get("VarCharValue")
        for item in rows[0]["Data"]
    ]

    data = [
        [
            item.get("VarCharValue")
            for item in row["Data"]
        ]
        for row in rows[1:]
    ]

    return pd.DataFrame(data, columns=columns)

In [1]:
import json
import subprocess

import pandas as pd

GLUE_DATABASE = "retail_demand_forecasting_raw"

# Retrieve the raw table definitions from the AWS Glue Data Catalog.
result = subprocess.run(
    [
        "aws",
        "glue",
        "get-tables",
        "--database-name",
        GLUE_DATABASE,
        "--query",
        "TableList[].{Table:Name,Location:StorageDescriptor.Location,"
        "Classification:Parameters.classification,"
        "Columns:StorageDescriptor.Columns}",
        "--output",
        "json",
    ],
    capture_output=True,
    text=True,
    check=True,
)

glue_tables = json.loads(result.stdout)

print(f"Discovered tables: {len(glue_tables)}")
print([table["Table"] for table in glue_tables])

Discovered tables: 5
['holidays_events', 'oil', 'stores', 'train', 'transactions']


In [2]:
# Flatten Glue table schemas into a tabular representation.
schema_rows = []

for table in glue_tables:
    for column in table["Columns"]:
        schema_rows.append(
            {
                "table": table["Table"],
                "column": column["Name"],
                "glue_type": column["Type"],
                "classification": table["Classification"],
                "location": table["Location"],
            }
        )

glue_schema = pd.DataFrame(schema_rows)

display(glue_schema)

,table,column,glue_type,classification,location
0,holidays_events,date,string,csv,s3://retail-demand-forecasting-platform-aws-za...
1,holidays_events,type,string,csv,s3://retail-demand-forecasting-platform-aws-za...
2,holidays_events,locale,string,csv,s3://retail-demand-forecasting-platform-aws-za...
3,holidays_events,locale_name,string,csv,s3://retail-demand-forecasting-platform-aws-za...
4,holidays_events,description,string,csv,s3://retail-demand-forecasting-platform-aws-za...
5,holidays_events,transferred,boolean,csv,s3://retail-demand-forecasting-platform-aws-za...
6,oil,date,string,csv,s3://retail-demand-forecasting-platform-aws-za...
7,oil,dcoilwtico,double,csv,s3://retail-demand-forecasting-platform-aws-za...
8,stores,store_nbr,bigint,csv,s3://retail-demand-forecasting-platform-aws-za...
9,stores,city,string,csv,s3://retail-demand-forecasting-platform-aws-za...


In [3]:
EXPECTED_TABLES = {
    "train",
    "stores",
    "oil",
    "holidays_events",
    "transactions",
}

actual_tables = {table["Table"] for table in glue_tables}

assert actual_tables == EXPECTED_TABLES

print("Glue table coverage validation passed.")

Glue table coverage validation passed.


In [5]:
EXPECTED_SCHEMAS = {
    "train": {
        "id": "bigint",
        "date": "string",
        "store_nbr": "bigint",
        "family": "string",
        "sales": "double",
        "onpromotion": "bigint",
    },
    "stores": {
        "store_nbr": "bigint",
        "city": "string",
        "state": "string",
        "type": "string",
        "cluster": "bigint",
    },
    "oil": {
        "date": "string",
        "dcoilwtico": "double",
    },
    "holidays_events": {
        "date": "string",
        "type": "string",
        "locale": "string",
        "locale_name": "string",
        "description": "string",
        "transferred": "boolean",
    },
    "transactions": {
        "date": "string",
        "store_nbr": "bigint",
        "transactions": "bigint",
    },
}

# Build the actual schemas returned by the Glue Data Catalog.
actual_schemas = {
    table["Table"]: {
        column["Name"]: column["Type"]
        for column in table["Columns"]
    }
    for table in glue_tables
}

# Compare all expected and actual schemas.
validation_rows = []

for table_name, expected_schema in EXPECTED_SCHEMAS.items():
    actual_schema = actual_schemas.get(table_name)

    validation_rows.append(
        {
            "table": table_name,
            "expected_columns": len(expected_schema),
            "actual_columns": len(actual_schema) if actual_schema is not None else 0,
            "schema_match": actual_schema == expected_schema,
        }
    )

schema_validation = pd.DataFrame(validation_rows)

display(schema_validation)

assert all(schema_validation["schema_match"]), (
    "At least one Glue table schema does not match the expected raw schema."
)

print("All Glue raw-table schema validations passed.")

,table,expected_columns,actual_columns,schema_match
0,train,6,6,True
1,stores,5,5,True
2,oil,2,2,True
3,holidays_events,6,6,True
4,transactions,3,3,True


All Glue raw-table schema validations passed.


## Raw Glue Catalog validation

The Glue crawler discovered the five expected raw datasets:

- `train`
- `stores`
- `oil`
- `holidays_events`
- `transactions`

The inferred schemas are consistent with the empirical raw-data inspection performed in `01_raw_data_exploration.ipynb`.

Notable observations:

- integer-valued columns were inferred as `bigint`;
- continuous numeric columns were inferred as `double`;
- `transferred` was correctly inferred as `boolean`;
- all raw date columns were inferred as `string`, so explicit date casting will be required in the transformation layer;
- `cluster` is physically stored as an integer but represents a categorical store attribute.

In [6]:
# Retrieve crawler metadata for the raw train table.
result = subprocess.run(
    [
        "aws",
        "glue",
        "get-table",
        "--database-name",
        GLUE_DATABASE,
        "--name",
        "train",
        "--query",
        (
            "Table.{"
            "Parameters:Parameters,"
            "Serde:StorageDescriptor.SerdeInfo,"
            "InputFormat:StorageDescriptor.InputFormat"
            "}"
        ),
        "--output",
        "json",
    ],
    capture_output=True,
    text=True,
    check=True,
)

train_metadata = json.loads(result.stdout)

parameters = train_metadata["Parameters"]
serde_parameters = train_metadata["Serde"]["Parameters"]

crawler_record_count = int(parameters["recordCount"])
validated_train_rows = 3_000_888

metadata_summary = pd.DataFrame(
    [
        {
            "classification": parameters["classification"],
            "delimiter": parameters["delimiter"],
            "serde_field_delimiter": serde_parameters["field.delim"],
            "skip_header_lines": int(parameters["skip.header.line.count"]),
            "object_count": int(parameters["objectCount"]),
            "crawler_record_count": crawler_record_count,
            "validated_raw_rows": validated_train_rows,
            "difference": crawler_record_count - validated_train_rows,
            "relative_difference_pct": (
                crawler_record_count / validated_train_rows - 1
            ) * 100,
        }
    ]
)

display(metadata_summary)

,classification,delimiter,serde_field_delimiter,skip_header_lines,object_count,crawler_record_count,validated_raw_rows,difference,relative_difference_pct
0,csv,",",",",1,1,3582363,3000888,581475,19.376764


### Glue crawler metadata

The crawler correctly identified the CSV format, comma delimiter, and one-line header.

The crawler also reports an estimated `recordCount` of 3,582,363 rows for `train`. This does not match the 3,000,888 rows observed during the raw-data exploration.

Glue crawler `recordCount` is an estimated metadata property rather than an exact table count. Exact row-count validation must therefore be performed by querying the data itself rather than relying on crawler statistics.

In [7]:
import json
import subprocess
import time

ATHENA_OUTPUT = (
    "s3://retail-demand-forecasting-platform-aws-zaamine/"
    "artifacts/athena-query-results/"
)

query = """
SELECT COUNT(*) AS row_count
FROM retail_demand_forecasting_raw.train
"""

# Start the Athena query.
result = subprocess.run(
    [
        "aws",
        "athena",
        "start-query-execution",
        "--query-string",
        query,
        "--query-execution-context",
        f"Database={GLUE_DATABASE}",
        "--result-configuration",
        f"OutputLocation={ATHENA_OUTPUT}",
        "--output",
        "json",
    ],
    capture_output=True,
    text=True,
    check=True,
)

query_execution_id = json.loads(result.stdout)["QueryExecutionId"]

# Wait until the query reaches a terminal state.
while True:
    status_result = subprocess.run(
        [
            "aws",
            "athena",
            "get-query-execution",
            "--query-execution-id",
            query_execution_id,
            "--query",
            "QueryExecution.Status.State",
            "--output",
            "text",
        ],
        capture_output=True,
        text=True,
        check=True,
    )

    query_status = status_result.stdout.strip()

    if query_status in {"SUCCEEDED", "FAILED", "CANCELLED"}:
        break

    time.sleep(1)

if query_status != "SUCCEEDED":
    raise RuntimeError(
        f"Athena query ended with status: {query_status}"
    )

# Retrieve the actual query result.
result = subprocess.run(
    [
        "aws",
        "athena",
        "get-query-results",
        "--query-execution-id",
        query_execution_id,
        "--output",
        "json",
    ],
    capture_output=True,
    text=True,
    check=True,
)

athena_result = json.loads(result.stdout)

athena_row_count = int(
    athena_result["ResultSet"]["Rows"][1]["Data"][0]["VarCharValue"]
)

print("Athena train row count:", athena_row_count)

Athena train row count: 3000888


In [8]:
EXPECTED_TRAIN_ROWS = 3_000_888

assert athena_row_count == EXPECTED_TRAIN_ROWS

print("Athena raw train row-count validation passed.")

Athena raw train row-count validation passed.


In [15]:
# Check whether the null values already exist when Athena reads the raw CSV table.
raw_sales_parse_issues = run_athena_query(
    """
    SELECT
        family,
        COUNT(*) AS affected_rows,
        SUM(CASE WHEN sales IS NULL THEN 1 ELSE 0 END) AS null_sales,
        SUM(CASE WHEN onpromotion IS NULL THEN 1 ELSE 0 END) AS null_onpromotion
    FROM retail_demand_forecasting_raw.train
    WHERE sales IS NULL
       OR onpromotion IS NULL
    GROUP BY family
    ORDER BY affected_rows DESC
    """
)

display(raw_sales_parse_issues)

,family,affected_rows,null_sales,null_onpromotion
0,"""LIQUOR",90936,90936,90936


### Raw CSV parsing issue

The automated dbt tests revealed 90,936 null values in both `sales` and
`onpromotion`, despite the local raw-data exploration showing no missing
values in these columns.

Investigation showed that all affected observations belong to the
`LIQUOR,WINE,BEER` product family:

- the family contains exactly 90,936 rows (`54 stores × 1,684 dates`);
- the source CSV correctly encloses this value in double quotes;
- the Glue crawler configured the raw table with `LazySimpleSerDe`;
- Athena therefore split the quoted family value at its internal commas,
  shifting the remaining fields and producing null numeric values.

The issue is therefore caused by the raw-table CSV parser rather than by
the source data or the dbt transformation.

In [16]:
# Verify that the corrected raw CSV parser no longer corrupts quoted family values.
raw_sales_parse_issues_after_fix = run_athena_query(
    """
    SELECT
        COUNT(*) AS malformed_rows,
        SUM(CASE WHEN sales IS NULL THEN 1 ELSE 0 END) AS null_sales,
        SUM(CASE WHEN onpromotion IS NULL THEN 1 ELSE 0 END) AS null_onpromotion
    FROM retail_demand_forecasting_raw.train
    WHERE sales IS NULL
       OR onpromotion IS NULL
    """
)

display(raw_sales_parse_issues_after_fix)

,malformed_rows,null_sales,null_onpromotion
0,0,None,None


In [17]:
# Confirm that the comma-containing family is now parsed as one complete field.
liquor_family_validation = run_athena_query(
    """
    SELECT
        family,
        COUNT(*) AS row_count,
        SUM(CASE WHEN sales IS NULL THEN 1 ELSE 0 END) AS null_sales,
        SUM(CASE WHEN onpromotion IS NULL THEN 1 ELSE 0 END) AS null_onpromotion
    FROM retail_demand_forecasting_raw.train
    WHERE family = 'LIQUOR,WINE,BEER'
    GROUP BY family
    """
)

display(liquor_family_validation)

,family,row_count,null_sales,null_onpromotion
0,"LIQUOR,WINE,BEER",90936,0,0


### Raw data query validation

Athena successfully queries the raw `train` dataset through the Glue Data Catalog and returns exactly 3,000,888 rows, matching the independent local validation performed in `01_raw_data_exploration.ipynb`.

The Glue crawler metadata reports an estimated `recordCount` of 3,582,363 rows, demonstrating that crawler statistics should not be used as exact row-count validation.

Exact data-quality checks must be performed against the underlying data.

In [2]:
# Validate that staging preserves the core forecasting panel structure.
curated_sales_summary = run_athena_query(
    """
    SELECT
        COUNT(*) AS row_count,
        COUNT(DISTINCT store_nbr) AS store_count,
        COUNT(DISTINCT family) AS family_count,
        COUNT(
            DISTINCT CAST(store_nbr AS VARCHAR) || '|' || family
        ) AS series_count,
        MIN(date) AS min_date,
        MAX(date) AS max_date,
        SUM(CASE WHEN date IS NULL THEN 1 ELSE 0 END) AS null_dates
    FROM retail_demand_forecasting_curated.stg_sales
    """
)

display(curated_sales_summary)

,row_count,store_count,family_count,series_count,min_date,max_date,null_dates
0,3000888,54,33,1782,2013-01-01,2017-08-15,0


In [3]:
summary = curated_sales_summary.iloc[0]

assert int(summary["row_count"]) == 3_000_888
assert int(summary["store_count"]) == 54
assert int(summary["family_count"]) == 33
assert int(summary["series_count"]) == 1_782
assert summary["min_date"] == "2013-01-01"
assert summary["max_date"] == "2017-08-15"
assert int(summary["null_dates"]) == 0

print("Curated sales structural validation passed.")

Curated sales structural validation passed.


In [4]:
# Validate the sales business grain: one row per date-store-family combination.
duplicate_keys = run_athena_query(
    """
    SELECT COUNT(*) AS duplicate_key_count
    FROM (
        SELECT
            date,
            store_nbr,
            family,
            COUNT(*) AS row_count
        FROM retail_demand_forecasting_curated.stg_sales
        GROUP BY
            date,
            store_nbr,
            family
        HAVING COUNT(*) > 1
    )
    """
)

display(duplicate_keys)

assert int(duplicate_keys.iloc[0]["duplicate_key_count"]) == 0

print("Curated sales business-grain validation passed.")

,duplicate_key_count
0,0


Curated sales business-grain validation passed.


In [5]:
curated_sales_schema = run_athena_query(
    """
    SELECT
        column_name,
        data_type
    FROM information_schema.columns
    WHERE table_schema = 'retail_demand_forecasting_curated'
      AND table_name = 'stg_sales'
    ORDER BY ordinal_position
    """
)

display(curated_sales_schema)

,column_name,data_type
0,id,bigint
1,date,date
2,store_nbr,bigint
3,family,varchar
4,sales,double
5,onpromotion,bigint


In [6]:
actual_types = dict(
    zip(
        curated_sales_schema["column_name"],
        curated_sales_schema["data_type"],
    )
)

assert actual_types["date"] == "date"

print("Curated sales date-type validation passed.")

Curated sales date-type validation passed.


### Curated sales validation

The `stg_sales` model was materialized by dbt as a Parquet-backed Athena table.

Independent Athena validation confirms that the transformation preserves the raw sales structure:

- 3,000,888 rows;
- 54 stores;
- 33 product families;
- 1,782 store-family series;
- date coverage from 2013-01-01 to 2017-08-15;
- no null dates;
- no duplicate `date × store_nbr × family` business keys.

The raw `date` field, originally catalogued as a string, is explicitly represented as a `date` in the curated layer.

In [13]:
# Compare curated row counts against the independently validated raw datasets.
curated_row_counts = run_athena_query(
    """
    SELECT 'stg_sales' AS table_name, COUNT(*) AS row_count
    FROM retail_demand_forecasting_curated.stg_sales

    UNION ALL

    SELECT 'stg_stores', COUNT(*)
    FROM retail_demand_forecasting_curated.stg_stores

    UNION ALL

    SELECT 'stg_oil', COUNT(*)
    FROM retail_demand_forecasting_curated.stg_oil

    UNION ALL

    SELECT 'stg_holidays_events', COUNT(*)
    FROM retail_demand_forecasting_curated.stg_holidays_events

    UNION ALL

    SELECT 'stg_transactions', COUNT(*)
    FROM retail_demand_forecasting_curated.stg_transactions
    """
)

display(curated_row_counts)

,table_name,row_count
0,stg_stores,54
1,stg_oil,1218
2,stg_holidays_events,350
3,stg_transactions,83488
4,stg_sales,3000888


In [8]:
# Expected counts were established during the raw-data exploration.
EXPECTED_ROW_COUNTS = {
    "stg_sales": 3_000_888,
    "stg_stores": 54,
    "stg_oil": 1_218,
    "stg_holidays_events": 350,
    "stg_transactions": 83_488,
}

actual_row_counts = {
    row["table_name"]: int(row["row_count"])
    for _, row in curated_row_counts.iterrows()
}

# Ensure that the raw-to-curated transformation neither dropped nor added rows.
assert actual_row_counts == EXPECTED_ROW_COUNTS

print("All curated row-count validations passed.")

All curated row-count validations passed.


In [9]:
# Retrieve the physical column types registered for all curated staging tables.
curated_schemas = run_athena_query(
    """
    SELECT
        table_name,
        column_name,
        data_type,
        ordinal_position
    FROM information_schema.columns
    WHERE table_schema = 'retail_demand_forecasting_curated'
      AND table_name IN (
          'stg_sales',
          'stg_stores',
          'stg_oil',
          'stg_holidays_events',
          'stg_transactions'
      )
    ORDER BY table_name, ordinal_position
    """
)

display(curated_schemas)

,table_name,column_name,data_type,ordinal_position
0,stg_holidays_events,date,date,1
1,stg_holidays_events,type,varchar,2
2,stg_holidays_events,locale,varchar,3
3,stg_holidays_events,locale_name,varchar,4
4,stg_holidays_events,description,varchar,5
5,stg_holidays_events,transferred,boolean,6
6,stg_oil,date,date,1
7,stg_oil,dcoilwtico,double,2
8,stg_sales,id,bigint,1
9,stg_sales,date,date,2


In [11]:
# Check uniqueness at the expected business grain of each staging table.
# Holidays are excluded because multiple legitimate events may occur on the same date.
key_validation = run_athena_query(
    """
    SELECT
        'stg_sales' AS table_name,
        COUNT(*) AS duplicate_key_groups
    FROM (
        SELECT date, store_nbr, family
        FROM retail_demand_forecasting_curated.stg_sales
        GROUP BY date, store_nbr, family
        HAVING COUNT(*) > 1
    )

    UNION ALL

    SELECT
        'stg_stores',
        COUNT(*)
    FROM (
        SELECT store_nbr
        FROM retail_demand_forecasting_curated.stg_stores
        GROUP BY store_nbr
        HAVING COUNT(*) > 1
    )

    UNION ALL

    SELECT
        'stg_oil',
        COUNT(*)
    FROM (
        SELECT date
        FROM retail_demand_forecasting_curated.stg_oil
        GROUP BY date
        HAVING COUNT(*) > 1
    )

    UNION ALL

    SELECT
        'stg_transactions',
        COUNT(*)
    FROM (
        SELECT date, store_nbr
        FROM retail_demand_forecasting_curated.stg_transactions
        GROUP BY date, store_nbr
        HAVING COUNT(*) > 1
    )
    """
)

display(key_validation)

# Every tested business key must remain unique after staging transformations.
assert (key_validation["duplicate_key_groups"].astype(int) == 0).all()

print("All curated business-key validations passed.")

,table_name,duplicate_key_groups
0,stg_stores,0
1,stg_oil,0
2,stg_transactions,0
3,stg_sales,0


All curated business-key validations passed.


In [12]:
# Verify that staging preserves known source-data characteristics rather than
# silently imputing or altering them.
preserved_raw_properties = run_athena_query(
    """
    SELECT
        (SELECT COUNT(*)
         FROM retail_demand_forecasting_curated.stg_oil
         WHERE dcoilwtico IS NULL) AS missing_oil_values,

        (SELECT COUNT(*)
         FROM retail_demand_forecasting_curated.stg_holidays_events
         WHERE transferred = true) AS transferred_holidays
    """
)

display(preserved_raw_properties)

# These values were independently observed in the raw-data exploration.
assert int(preserved_raw_properties.iloc[0]["missing_oil_values"]) == 43
assert int(preserved_raw_properties.iloc[0]["transferred_holidays"]) == 12

print("Raw-data semantics preserved in the curated layer.")

,missing_oil_values,transferred_holidays
0,43,12


Raw-data semantics preserved in the curated layer.


### Curated layer validation

All five raw datasets were successfully materialized as typed Parquet staging tables.

Validation against the independently explored raw datasets confirms that:

- row counts are preserved for all five sources;
- date fields are explicitly typed as `date`;
- expected business keys remain unique for sales, stores, oil and transactions;
- source-specific semantics are preserved, including the 43 missing oil-price observations and 12 transferred holidays;
- no feature engineering or missing-value imputation has yet been applied.

The curated staging layer therefore provides a typed and reproducible representation of the raw Favorita datasets while preserving their original business semantics.

## Automated data-quality tests

The exploratory validations above were converted into automated dbt data tests so that critical data contracts can be checked on every pipeline run.

The staging layer currently enforces:

- non-null values for required business fields;
- uniqueness of technical/store/date keys where appropriate;
- uniqueness of the `date × store_nbr × family` sales grain;
- uniqueness of the `date × store_nbr` transaction grain;
- absence of fully duplicated holiday/event records.

The first test run detected 90,936 unexpected null values in both `sales` and `onpromotion`.

Investigation traced the issue to the Glue crawler's initial CSV deserializer: the `LIQUOR,WINE,BEER` family contains embedded commas and requires quote-aware CSV parsing. After replacing `LazySimpleSerDe` with `OpenCSVSerde`, rebuilding `stg_sales`, and rerunning the tests, all 27 data-quality tests passed.

This incident demonstrates why matching row counts alone is insufficient to validate a data pipeline.